# 01 — RAGAs Metrics em Profundidade

Como avaliar um sistema RAG objetivamente?
RAGAs oferece 4 metricas que cobrem os principais pontos de falha.

## As 4 Metricas

```
Query ──→ [Retriever] ──→ [Generator] ──→ Resposta
                |               |
    context_precision    faithfulness
    context_recall       answer_relevancy
```

| Metrica | Escopo | O que mede | Calculo |
|---------|--------|-----------|----------|
| **Faithfulness** | Generator | Resposta e factualmente correta? | LLM extrai afirmacoes → verifica no contexto |
| **Answer Relevancy** | Generator | Resposta responde a pergunta? | Gera queries alternativas → cosine com original |
| **Context Precision** | Retriever | Docs recuperados sao relevantes? | LLM classifica cada doc como relevante/nao |
| **Context Recall** | Retriever | Recuperou todo o necessario? | Compara afirmacoes da GT com contexto |

**RAGAs Score = media(faithfulness, answer_relevancy, context_precision, context_recall)**

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ollama
import httpx
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

try:
    r = httpx.get('http://localhost:11434/api/tags')
    modelos = [m['name'] for m in r.json().get('models', [])]
    LLM = 'llama3.2' if any('llama3.2' in m for m in modelos) else (modelos[0] if modelos else None)
    print(f'LLM: {LLM}')
except:
    LLM = None

print('Modulos carregados!')

## 6.1 Faithfulness: A resposta e suportada pelo contexto?

**Alto faithfulness** = LLM nao alucina, responde apenas com o que esta no contexto.

```
1. Extrair todas as afirmacoes da resposta
2. Para cada afirmacao: ela pode ser deduzida do contexto?
3. Faithfulness = afirmacoes_suportadas / total_afirmacoes
```

In [ ]:
# Implementacao simplificada do calculo de faithfulness

EXTRACT_CLAIMS_PROMPT = """Extraia as afirmacoes factuais da resposta abaixo.
Retorne APENAS as afirmacoes, uma por linha, sem numeracao.

Resposta:
{resposta}

Afirmacoes:"""

VERIFY_CLAIM_PROMPT = """A afirmacao abaixo pode ser deduzida do contexto fornecido?
Responda apenas: SIM ou NAO

Contexto:
{contexto}

Afirmacao: {afirmacao}

Resposta (SIM/NAO):"""

def calcular_faithfulness(resposta, contextos, verbose=True):
    if not LLM:
        return 0.0, []
    
    contexto_str = '\n'.join(contextos)
    
    # 1. Extrair afirmacoes
    prompt = EXTRACT_CLAIMS_PROMPT.format(resposta=resposta)
    claims_response = ollama.chat(model=LLM, messages=[{'role': 'user', 'content': prompt}])
    claims = [c.strip() for c in claims_response['message']['content'].split('\n') if c.strip()]
    
    if verbose:
        print(f'Afirmacoes extraidas ({len(claims)}):')
        for c in claims:
            print(f'  - {c}')
    
    # 2. Verificar cada afirmacao
    supported = []
    for claim in claims:
        prompt_verify = VERIFY_CLAIM_PROMPT.format(contexto=contexto_str[:2000], afirmacao=claim)
        verify_response = ollama.chat(model=LLM, messages=[{'role': 'user', 'content': prompt_verify}])
        is_supported = 'SIM' in verify_response['message']['content'].upper()
        supported.append(is_supported)
        if verbose:
            print(f'  {"OK" if is_supported else "X":2s}: {claim[:60]}')
    
    score = sum(supported) / len(supported) if supported else 0.0
    return score, supported

# Exemplo de teste
contexto_teste = [
    'HNSW usa m=16 como valor padrao para conexoes por no.',
    'O parametro ef_construct define os candidatos durante a construcao, com default 100.',
]

# Resposta boa (alta faithfulness)
resposta_boa = 'HNSW usa m=16 por padrao para conexoes por no, e ef_construct=100 para candidatos na construcao.'

# Resposta com alucinacao (baixa faithfulness)
resposta_alucinada = 'HNSW usa m=16 por padrao. Alem disso, o algoritmo foi criado em 2015 por pesquisadores russos e e o mais rapido do mundo.'

print('=== RESPOSTA BOA (esperado: alta faithfulness) ===')
score_boa, _ = calcular_faithfulness(resposta_boa, contexto_teste)
print(f'Faithfulness: {score_boa:.2f}\n')

print('=== RESPOSTA COM ALUCINACAO (esperado: baixa faithfulness) ===')
score_alucin, _ = calcular_faithfulness(resposta_alucinada, contexto_teste)
print(f'Faithfulness: {score_alucin:.2f}')

## 6.2 Answer Relevancy: A resposta responde a pergunta?

```
1. Gerar n perguntas alternativas a partir da resposta
2. Calcular similaridade cosine entre perguntas geradas e original
3. Answer Relevancy = media das similaridades
```

In [ ]:
GEN_QUESTIONS_PROMPT = """Dada a resposta abaixo, gere {n} perguntas que essa resposta responderia diretamente.
Retorne APENAS as perguntas, uma por linha, sem numeracao.

Resposta: {resposta}

Perguntas:"""

def calcular_answer_relevancy(pergunta_original, resposta, n_perguntas=3, verbose=True):
    if not LLM:
        return 0.0
    
    prompt = GEN_QUESTIONS_PROMPT.format(resposta=resposta, n=n_perguntas)
    response = ollama.chat(model=LLM, messages=[{'role': 'user', 'content': prompt}])
    generated_qs = [q.strip() for q in response['message']['content'].split('\n') if q.strip()]
    
    if verbose:
        print(f'Perguntas geradas a partir da resposta:')
        for q in generated_qs:
            print(f'  - {q}')
    
    # Calcular similaridade com a pergunta original
    original_emb = embed_model.encode(pergunta_original, normalize_embeddings=True)
    generated_embs = embed_model.encode(generated_qs, normalize_embeddings=True)
    
    similarities = generated_embs @ original_emb
    score = float(np.mean(similarities))
    
    if verbose:
        for q, s in zip(generated_qs, similarities):
            print(f'  sim={s:.3f}: {q[:60]}')
    
    return score

# Teste
pergunta = 'Quais sao os parametros do HNSW?'

resposta_relevante = 'Os parametros principais do HNSW sao m (conexoes por no) e ef_construct (candidatos na construcao).'
resposta_irrelevante = 'Python e uma linguagem de programacao muito popular para data science e machine learning.'

print(f'Pergunta original: {pergunta}')
print('\n=== RESPOSTA RELEVANTE ===')
score_rel = calcular_answer_relevancy(pergunta, resposta_relevante)
print(f'Answer Relevancy: {score_rel:.3f}')

print('\n=== RESPOSTA IRRELEVANTE ===')
score_irrel = calcular_answer_relevancy(pergunta, resposta_irrelevante)
print(f'Answer Relevancy: {score_irrel:.3f}')

## 6.3 Context Precision e Recall

**Context Precision**: Qual percentual dos docs recuperados e realmente relevante?
```
Context Precision = docs_relevantes_recuperados / total_docs_recuperados
```

**Context Recall**: Recuperou toda a informacao necessaria para responder?
```
Context Recall = afirmacoes_encontradas_no_contexto / total_afirmacoes_ground_truth
```

In [ ]:
# Visualizacao: como cada metrica diagnostica problemas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

cenarios = [
    {
        'nome': 'Sistema Ideal',
        'faithfulness': 0.95,
        'answer_relevancy': 0.92,
        'context_precision': 0.88,
        'context_recall': 0.90,
        'cor': '#2ecc71',
    },
    {
        'nome': 'LLM Alucinando',
        'faithfulness': 0.45,  # BAIXO
        'answer_relevancy': 0.88,
        'context_precision': 0.85,
        'context_recall': 0.87,
        'cor': '#e74c3c',
    },
    {
        'nome': 'Retrieval Ruim',
        'faithfulness': 0.85,
        'answer_relevancy': 0.60,  # BAIXO
        'context_precision': 0.40, # BAIXO
        'context_recall': 0.55,    # BAIXO
        'cor': '#f39c12',
    },
    {
        'nome': 'Chunking Ruim',
        'faithfulness': 0.80,
        'answer_relevancy': 0.78,
        'context_precision': 0.82,
        'context_recall': 0.35,    # BAIXO (nao recupera tudo)
        'cor': '#9b59b6',
    },
]

metricas = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
labels = ['Faithfulness', 'Answ. Relevancy', 'Ctx Precision', 'Ctx Recall']

for ax, cenario in zip(axes.flatten(), cenarios):
    scores = [cenario[m] for m in metricas]
    cores = ['#2ecc71' if s >= 0.8 else '#f39c12' if s >= 0.6 else '#e74c3c' for s in scores]
    
    bars = ax.bar(labels, scores, color=cores, edgecolor='white', linewidth=2)
    ax.axhline(y=0.8, color='darkgreen', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_ylim(0, 1.1)
    ax.set_title(cenario['nome'], fontsize=12, fontweight='bold', color=cenario['cor'])
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{score:.2f}', ha='center', fontweight='bold', fontsize=10)

plt.suptitle('Diagnostico de Problemas via RAGAs Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nDiagnostico:')
print('  Faithfulness baixo → LLM alucinando → Prompt mais restritivo')
print('  Answer Relevancy baixo → Retrieval de docs errados → Melhorar embedding')
print('  Context Precision baixo → Muitos docs irrelevantes → Re-ranking, threshold')
print('  Context Recall baixo → Perdendo docs importantes → Maior k, melhor chunking')

## Usando RAGAs com Ollama Local

RAGAs suporta LLMs e embeddings locais via LangChain.

In [ ]:
# Template de uso do RAGAs com Ollama
print('''
Como usar RAGAs com Ollama local:

```python
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset
from langchain_ollama import ChatOllama
from langchain_ollama.embeddings import OllamaEmbeddings

# Dataset com formato RAGAs
data = {
    "question": ["O que e HNSW?"],
    "answer": ["HNSW e um algoritmo de indexacao..."],
    "contexts": [["HNSW (Hierarchical...)", "O parametro m..."]],
    "ground_truth": ["HNSW e um algoritmo para busca ANN..."]
}
dataset = Dataset.from_dict(data)

# LLM local via Ollama
llm = ChatOllama(model="llama3.2", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Avaliar
results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=llm,
    embeddings=embeddings,
)

print(results)
# Output: {'faithfulness': 0.87, 'answer_relevancy': 0.92, ...}
```
''')

print('\nDica: Execute este template no 05_pocs/evaluation_ragas/evaluate.ipynb')

## Resumo Final

### As 4 Metricas em Uma Frase

- **Faithfulness**: "O LLM so disse o que o contexto suporta?"
- **Answer Relevancy**: "A resposta realmente responde a pergunta?"
- **Context Precision**: "Os documentos recuperados eram necessarios?"
- **Context Recall**: "Recuperamos todos os documentos necessarios?"

### Score Target
- **> 0.8**: Sistema pronto para producao
- **0.6 - 0.8**: Adequado, mas com espaco para melhoria
- **< 0.6**: Problemas serios, requer revisao do pipeline

### Ciclo de Melhoria

```
1. Rodar RAGAs → identificar metrica mais baixa
2. Diagnosticar causa raiz (chunking? embedding? prompt? LLM?)
3. Implementar melhoria
4. Re-rodar RAGAs → verificar melhoria
5. Repeat
```

**Parabens!** Voce completou o playground de RAG e Embeddings.

Revise os READMEs de cada modulo para consolidar o conhecimento:
- [01 Embeddings](../01_embeddings/README.md)
- [02 Vector DBs](../02_vector_databases/README.md)
- [03 RAG Fundamentals](../03_rag_fundamentals/README.md)
- [04 Arquiteturas](../04_rag_architectures/README.md)
- [docs/rag/architectures.md](../docs/rag/architectures.md)